In [1]:
notebookutils.fs.ls("Files/")

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 3, Finished, Available, Finished, False)

[FileInfo(path=abfss://cb1d745a-a784-46f8-aa7a-a04d520ff8d1@onelake.dfs.fabric.microsoft.com/ecd0b5e7-24d2-4769-9077-6cd9fed06372/Files/Influencer Users Report (8).csv, name=Influencer Users Report (8).csv, size=247850004),
 FileInfo(path=abfss://cb1d745a-a784-46f8-aa7a-a04d520ff8d1@onelake.dfs.fabric.microsoft.com/ecd0b5e7-24d2-4769-9077-6cd9fed06372/Files/club_report_sample_synthetic.csv, name=club_report_sample_synthetic.csv, size=332034)]

In [4]:
file_path = "Files/club_report_sample_synthetic.csv"
df = spark.read.option("header",True).option("inferSchema",True).csv(file_path)
print("Total rowa: ", df.count())
df.show(5,truncate= False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 8, Finished, Available, Finished, False)

Total rowa:  2000
+------------------+-----------------+----------------------+--------+-----------+--------------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|External State    |External District|External District Code|SAP_CODE|HUMBEE_CODE|FIRM_NAME     |FIRM_TYPE|SCHEME_PERIOD_START_DATE|SCHEME_PERIOD_END_DATE|SCHEME_PERIOD_MONTH|CLUB_NAME|SCHEME_PERIOD_SALES_AVG|ELIGIBLE_FOR_BENEFIT|ELIGIBLE_STATUS_MESSAGE|SALES_CLUB_RANK|POLICY_CLUB_RANK|POLICY_CLUB_NAME|
+------------------+-----------------+----------------------+--------+-----------+--------------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|Sample State North|Demo District A  |D04                   |900000

In [5]:
from pyspark.sql import functions as F 
#Count missing value in every column
df.select(
    [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in df.columns
    ]
).show(truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 9, Finished, Available, Finished, False)

+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|External State|External District|External District Code|SAP_CODE|HUMBEE_CODE|FIRM_NAME|FIRM_TYPE|SCHEME_PERIOD_START_DATE|SCHEME_PERIOD_END_DATE|SCHEME_PERIOD_MONTH|CLUB_NAME|SCHEME_PERIOD_SALES_AVG|ELIGIBLE_FOR_BENEFIT|ELIGIBLE_STATUS_MESSAGE|SALES_CLUB_RANK|POLICY_CLUB_RANK|POLICY_CLUB_NAME|
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|0             |0                |0                     |0       |0          |0        |0        |0             

In [7]:
df.groupBy("ELIGIBLE_FOR_BENEFIT").agg(
    F.count("*").alias("Total_rows"),
    F.sum(
        F.col("ELIGIBLE_STATUS_MESSAGE").isnull().cast("int")
    ).alias("missing_status_message")
).show()

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 11, Finished, Available, Finished, False)

TypeError: 'Column' object is not callable

In [8]:
from pyspark.sql import functions as F

df.groupBy("ELIGIBLE_FOR_BENEFIT").agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.col("ELIGIBLE_STATUS_MESSAGE").isNull().cast("int")
    ).alias("missing_status_messages")
).show()

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 12, Finished, Available, Finished, False)

+--------------------+----------+-----------------------+
|ELIGIBLE_FOR_BENEFIT|total_rows|missing_status_messages|
+--------------------+----------+-----------------------+
|                true|      1003|                   1003|
|               false|       997|                      0|
+--------------------+----------+-----------------------+



In [11]:
duplicate_groups = (
    df.groupBy(
        "SAP_CODE",
        "SCHEME_PERIOD_END_DATE"
    )
    .count()
    .filter(F.col("count")>1)
)

print("Duplicates dealer period groups",duplicate_groups.count())
duplicate_groups.show(10,truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 18, Finished, Available, Finished, False)

Duplicates dealer period groups 0
+--------+----------------------+-----+
|SAP_CODE|SCHEME_PERIOD_END_DATE|count|
+--------+----------------------+-----+
+--------+----------------------+-----+



In [15]:
# Make column names safe for Delta tables
for old_name in df.columns:
    new_name = (
        old_name.strip()
        .lower()
        .replace(" ","_")
        .replace("-","_")
        .replace("/","_")
    )
    df = df.withColumnRenamed(old_name,new_name)

#Check the new column name 
print(df.columns)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 22, Finished, Available, Finished, False)

['external_state', 'external_district', 'external_district_code', 'sap_code', 'humbee_code', 'firm_name', 'firm_type', 'scheme_period_start_date', 'scheme_period_end_date', 'scheme_period_month', 'club_name', 'scheme_period_sales_avg', 'eligible_for_benefit', 'eligible_status_message', 'sales_club_rank', 'policy_club_rank', 'policy_club_name']


In [16]:
(
    df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable("Bronze_club_report") 
)

print("Bronze table record")

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 23, Finished, Available, Finished, False)

Bronze table record


In [17]:
bronze_df = spark.read.table("bronze_club_report")
print("Bronze row count : ", bronze_df.count())

bronze_df.show(5,truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 26, Finished, Available, Finished, False)

Bronze row count :  2000
+------------------+-----------------+----------------------+--------+-----------+--------------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|external_state    |external_district|external_district_code|sap_code|humbee_code|firm_name     |firm_type|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|sales_club_rank|policy_club_rank|policy_club_name|
+------------------+-----------------+----------------------+--------+-----------+--------------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|Sample State North|Demo District A  |D04                   

In [18]:
# Read the bronze table 
bronze_df = spark.read.table("bronze_club_report")

#check the column names and data types
bronze_df.printSchema()

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 27, Finished, Available, Finished, False)

root
 |-- external_state: string (nullable = true)
 |-- external_district: string (nullable = true)
 |-- external_district_code: string (nullable = true)
 |-- sap_code: integer (nullable = true)
 |-- humbee_code: integer (nullable = true)
 |-- firm_name: string (nullable = true)
 |-- firm_type: string (nullable = true)
 |-- scheme_period_start_date: date (nullable = true)
 |-- scheme_period_end_date: date (nullable = true)
 |-- scheme_period_month: string (nullable = true)
 |-- club_name: string (nullable = true)
 |-- scheme_period_sales_avg: double (nullable = true)
 |-- eligible_for_benefit: boolean (nullable = true)
 |-- eligible_status_message: string (nullable = true)
 |-- sales_club_rank: integer (nullable = true)
 |-- policy_club_rank: integer (nullable = true)
 |-- policy_club_name: string (nullable = true)



In [19]:
# check the data type of our sales column 
bronze_df.select("scheme_period_sales_avg").printSchema()

# Look at five sales_value
bronze_df.select("scheme_period_sales_avg").show(5)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 28, Finished, Available, Finished, False)

root
 |-- scheme_period_sales_avg: double (nullable = true)

+-----------------------+
|scheme_period_sales_avg|
+-----------------------+
|               2643.756|
|               2346.382|
|               2751.613|
|               2472.927|
|               3344.249|
+-----------------------+
only showing top 5 rows


In [22]:
from pyspark.sql import functions as F

negative_sales = bronze_df.filter(
    F.col("scheme_period_sales_avg")<0
)

print("Rows with the negative sales :",negative_sales.count())

negative_sales.show(5,truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 33, Finished, Available, Finished, False)

Rows with the negative sales : 0
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|external_state|external_district|external_district_code|sap_code|humbee_code|firm_name|firm_type|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|sales_club_rank|policy_club_rank|policy_club_name|
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
+--------------+-----------------+----------------------+--------+-----------+-

In [24]:
invalid_dates = bronze_df.filter(
    F.col("scheme_period_start_date").isNull()|
    F.col("scheme_period_end_date").isNull()|
    (
        F.col("scheme_period_end_date")<
        F.col("scheme_period_start_date")
    )
)

print("Rows with invalid Dates : ", invalid_dates.count())

invalid_dates.show(5,truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 37, Finished, Available, Finished, False)

Rows with invalid Dates :  0
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|external_state|external_district|external_district_code|sap_code|humbee_code|firm_name|firm_type|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|sales_club_rank|policy_club_rank|policy_club_name|
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
+--------------+-----------------+----------------------+--------+-----------+-----

In [25]:
missing_sap_code = bronze_df.filter(
    F.col("sap_code").isNull()|
    (F.trim(F.col("sap_code"))=="")

)

print("Rows with missing Sap Codes :",missing_sap_code.count())
missing_sap_code.show(5, truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 40, Finished, Available, Finished, False)

Rows with missing Sap Codes : 0
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
|external_state|external_district|external_district_code|sap_code|humbee_code|firm_name|firm_type|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|sales_club_rank|policy_club_rank|policy_club_name|
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+
+--------------+-----------------+----------------------+--------+-----------+--

In [29]:
# Silver Table
from pyspark.sql import functions as F 
from pyspark.sql.types import DecimalType, IntegerType

# read bronze table
silver_df = spark.read.table("bronze_club_report")

# Read our bronze table
silver_df = (
     silver_df
    .withColumn("sap_code",F.col("sap_code").cast("string"))
    .withColumn("humbee_code",F.col("humbee_code").cast("string"))
    .withColumn(
        "schemePeriod_sales_avg",
        F.col("scheme_period_sales_avg").cast(DecimalType(18,3))
    )
    .withColumn("sales_club_rank",F.col("sales_club_rank").cast(IntegerType()))
    .withColumn("policy_club_rank", F.col("policy_club_rank").cast(IntegerType()))

)
print("Silver preparation complete!")
print("Rows : ", silver_df.count())

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 44, Finished, Available, Finished, False)

Silver preparation complete!
Rows :  2000


In [32]:
# save silver table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_club_report")
)

print("silver table created sucessfully")

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 47, Finished, Available, Finished, False)

silver table created sucessfully


In [34]:
silver_check = spark.read.table("silver_club_report")

print("silver rows count :", silver_check.count())

silver_check.show(5,truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 51, Finished, Available, Finished, False)

silver rows count : 2000
+------------------+-----------------+----------------------+--------+-----------+--------------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+----------------------+
|external_state    |external_district|external_district_code|sap_code|humbee_code|firm_name     |firm_type|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|sales_club_rank|policy_club_rank|policy_club_name|schemePeriod_sales_avg|
+------------------+-----------------+----------------------+--------+-----------+--------------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+---------------

In [35]:
from pyspark.sql import functions as F 

invalid_sales = silver_check.filter( 
    F.col("scheme_period_sales_avg")<0
)

print("Rows with negative sales:", invalid_sales.count())
invalid_sales.show(5,truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 54, Finished, Available, Finished, False)

Rows with negative sales: 0
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+----------------------+
|external_state|external_district|external_district_code|sap_code|humbee_code|firm_name|firm_type|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|sales_club_rank|policy_club_rank|policy_club_name|schemePeriod_sales_avg|
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+----------------------+
+--------------

In [36]:
missing_firm = silver_check.filter(
    F.col("firm_name").isNull()|
    (F.trim(F.col("firm_name"))=="")
)

print("Rows with missing firm name :",missing_firm.count())
missing_firm.show(5,truncate=False)

StatementMeta(, edfe2236-48dc-4275-a00a-54a29c5ddfd8, 57, Finished, Available, Finished, False)

Rows with missing firm name : 0
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+----------------------+
|external_state|external_district|external_district_code|sap_code|humbee_code|firm_name|firm_type|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|sales_club_rank|policy_club_rank|policy_club_name|schemePeriod_sales_avg|
+--------------+-----------------+----------------------+--------+-----------+---------+---------+------------------------+----------------------+-------------------+---------+-----------------------+--------------------+-----------------------+---------------+----------------+----------------+----------------------+
+----------

In [1]:
# Gold Table 1 - How many firms are in each club?
from pyspark.sql import functions as F

silver_df = spark.table("silver_club_report")

(
    silver_df.groupBy("external_state","external_district","club_name")
    .agg(
        F.countDistinct("sap_code").alias('firm_count'),
        F.round(
            F.sum("scheme_period_sales_avg"),2
        ).alias("total_sales_avg"),
        F.count("*").alias("record_count")
        
    )
    .orderBy("external_state","external_district","club_name")
    .show(20,truncate=False)
)

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 3, Finished, Available, Finished, False)

+------------------+-----------------+---------+----------+---------------+------------+
|external_state    |external_district|club_name|firm_count|total_sales_avg|record_count|
+------------------+-----------------+---------+----------+---------------+------------+
|Sample State East |Demo District J  |BRONZE   |14        |71891.61       |56          |
|Sample State East |Demo District J  |SILVER   |26        |297770.05      |104         |
|Sample State East |Demo District K  |BRONZE   |11        |54120.67       |44          |
|Sample State East |Demo District K  |SILVER   |27        |346859.05      |108         |
|Sample State East |Demo District L  |BRONZE   |15        |69390.6        |60          |
|Sample State East |Demo District L  |SILVER   |32        |373317.68      |128         |
|Sample State North|Demo District A  |BRONZE   |17        |81611.51       |68          |
|Sample State North|Demo District A  |SILVER   |23        |266643.93      |92          |
|Sample State North|D

In [2]:
# turn the resuabel result into gold table
gold_club_summary = (
    silver_df
    .groupBy("external_state","external_district","club_name")
    .agg(
        F.countDistinct("sap_code").alias("firm_count"),
        F.round(
            F.sum("scheme_period_sales_avg"),2
        ).alias("total_sales_avg"),
        F.count("*").alias("record_count")
    )
)

(
    gold_club_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_club_summary")
)

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 4, Finished, Available, Finished, False)

In [3]:
# validating save table
spark.table("gold_club_summary").show(20,truncate=False)

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 5, Finished, Available, Finished, False)

+------------------+-----------------+---------+----------+---------------+------------+
|external_state    |external_district|club_name|firm_count|total_sales_avg|record_count|
+------------------+-----------------+---------+----------+---------------+------------+
|Sample State East |Demo District L  |SILVER   |32        |373317.68      |128         |
|Sample State North|Demo District C  |SILVER   |29        |334067.33      |116         |
|Sample State North|Demo District B  |SILVER   |28        |346995.05      |112         |
|Sample State South|Demo District I  |SILVER   |30        |286386.68      |120         |
|Sample State East |Demo District L  |BRONZE   |15        |69390.6        |60          |
|Sample State North|Demo District A  |BRONZE   |17        |81611.51       |68          |
|Sample State West |Demo District D  |BRONZE   |22        |84328.61       |88          |
|Sample State South|Demo District H  |BRONZE   |19        |90320.29       |76          |
|Sample State South|D

In [5]:
silver_df.printSchema()
silver_df.show(5,truncate=False)


StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 9, Finished, Available, Finished, False)

root
 |-- external_state: string (nullable = true)
 |-- external_district: string (nullable = true)
 |-- external_district_code: string (nullable = true)
 |-- sap_code: string (nullable = true)
 |-- humbee_code: string (nullable = true)
 |-- firm_name: string (nullable = true)
 |-- firm_type: string (nullable = true)
 |-- scheme_period_start_date: date (nullable = true)
 |-- scheme_period_end_date: date (nullable = true)
 |-- scheme_period_month: string (nullable = true)
 |-- club_name: string (nullable = true)
 |-- scheme_period_sales_avg: double (nullable = true)
 |-- eligible_for_benefit: boolean (nullable = true)
 |-- eligible_status_message: string (nullable = true)
 |-- sales_club_rank: integer (nullable = true)
 |-- policy_club_rank: integer (nullable = true)
 |-- policy_club_name: string (nullable = true)
 |-- schemePeriod_sales_avg: decimal(18,3) (nullable = true)

+------------------+-----------------+----------------------+--------+-----------+--------------+---------+------

In [9]:
from pyspark.sql import functions as F 

#Read the existing silver Table
silver_df = spark.table("silver_club_report")

# Build a bussiness-ready benefit status table 
gold_firm_benefit_status = (
    silver_df 
    .select(
        "sap_code",
        "humbee_code",
        "firm_name",
        
    )
)

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 13, Finished, Available, Finished, False)

In [14]:
#Read existing silver table
silver_df = spark.table("silver_club_report")

#build a bussiness ready benefir status table
gold_firm_benefit_status = (
    silver_df
    .select(
        "sap_code",
        "humbee_code",
        "firm_name",
        "firm_type",
        "external_state",
        "external_district",
        "scheme_period_start_date",
        "scheme_period_end_date",
        "scheme_period_month",
        "club_name",
        "policy_club_name",
        "scheme_period_sales_avg",
        "eligible_for_benefit",
        "eligible_status_message"
    )
    .withColumn(
        "benefit_status",
        F.when(
            F.col("eligible_for_benefit")== True,
            F.lit("Eligible")
        )
        .when(
            F.col("eligible_for_benefit")==False,
            F.lit("Not Eligible")
        )
        .otherwise(F.lit("Unknown"))
    )
    .withColumn(
        "benefit_status_reason",
        F.when(
            F.col("eligible_for_benefit").isNull(),
            F.lit("Eligible not available")
        )
        .when(
            F.col("eligible_for_benefit")==True,
            F.lit("Eligible undeer record rules")
        )
        .when(
            F.col("eligible_status_message").isNotNull(),
            F.col("eligible_status_message")
        )
        .otherwise(F.lit("Reason not provided in sources"))
    )
) 


#Save the gold table
(
    gold_firm_benefit_status.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_firm_benefit_status")
)

print("Gold table 2 created successfully")

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 18, Finished, Available, Finished, False)

Gold table 2 created successfully


In [16]:
gold_df = spark.table("gold_firm_benefit_status")

#check sample record
gold_df.show(5,truncate=False)

# count eligible, ineligible and unkown records
gold_df.groupBy("benefit_status").count().show()

# Investigating the recorded status message
gold_df.groupBy(
    "eligible_for_benefit",
    "eligible_status_message"
).count().show(truncate=False)

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 22, Finished, Available, Finished, False)

+--------+-----------+--------------+---------+------------------+-----------------+------------------------+----------------------+-------------------+---------+----------------+-----------------------+--------------------+-----------------------+--------------+----------------------------+
|sap_code|humbee_code|firm_name     |firm_type|external_state    |external_district|scheme_period_start_date|scheme_period_end_date|scheme_period_month|club_name|policy_club_name|scheme_period_sales_avg|eligible_for_benefit|eligible_status_message|benefit_status|benefit_status_reason       |
+--------+-----------+--------------+---------+------------------+-----------------+------------------------+----------------------+-------------------+---------+----------------+-----------------------+--------------------+-----------------------+--------------+----------------------------+
|9000001 |800001     |Demo Firm 0001|DEALER   |Sample State North|Demo District A  |2025-11-01              |2025-12-31  

In [19]:
(
    gold_df
    .filter(F.col("benefit_status")=="Not Eligible")
    .select(
        "sap_code",
        "firm_name",
        "scheme_period_month",
        "club_name",
        "policy_club_name",
        "eligible_status_message",
        "benefit_status_reason"
    )
    .show(5,truncate=False)
)

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 25, Finished, Available, Finished, False)

+--------+--------------+-------------------+---------+----------------+----------------------------------+----------------------------------+
|sap_code|firm_name     |scheme_period_month|club_name|policy_club_name|eligible_status_message           |benefit_status_reason             |
+--------+--------------+-------------------+---------+----------------+----------------------------------+----------------------------------+
|9000003 |Demo Firm 0003|Apr-26-Jun-26      |SILVER   |BRONZE          |Illustrative policy check failed  |Illustrative policy check failed  |
|9000004 |Demo Firm 0004|Jan-26-Mar-26      |SILVER   |BRONZE          |Sales below illustrative threshold|Sales below illustrative threshold|
|9000004 |Demo Firm 0004|Apr-26-Jun-26      |SILVER   |BRONZE          |Sales below illustrative threshold|Sales below illustrative threshold|
|9000004 |Demo Firm 0004|Jul-26-Sep-26      |SILVER   |BRONZE          |Sales below illustrative threshold|Sales below illustrative threshold|

In [20]:
# Gold table 3 : gold firm club histroy
silver_df = spark.table("silver_club_report")

gold_firm_club_history = (
    silver_df
    .select(
        "sap_code",
        "humbee_code",
        "firm_name",
        "firm_type",
        "external_state",
        "external_district",
        "scheme_period_start_date",
        "scheme_period_end_date",
        "scheme_period_month",
        "club_name",
        "policy_club_name",
        "policy_club_rank",
        "scheme_period_sales_avg",
        "eligible_for_benefit"

    )
    .dropDuplicates(
        [
            "sap_code",
            "scheme_period_start_date",
            "scheme_period_end_date",
            "club_name"
        ]
    )
)

(gold_firm_club_history.write
.format("delta")
.mode("overwrite")
.saveAsTable("gold_firm_club_history")
)

print("Gold Table 3 created successfully")

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 26, Finished, Available, Finished, False)

Gold Table 3 created successfully


In [22]:
history_df = spark.table("gold_firm_club_history")

gold_repeat_club_summary = (
    history_df
    .groupBy("sap_code","firm_name","club_name")
    .agg(
        F.countDistinct(
            "scheme_period_start_date"
        ).alias("periods_in_club"),
        F.min("scheme_period_start_date").alias("first_period"),
        F.max("scheme_period_end_date").alias("last_period"),
        F.countDistinct("eligible_for_benefit").alias(
            "elgibilty_values"
        )
    )
    .filter(F.col("periods_in_club")>=2)
    .orderBy(
        F.desc("periods_in_club"),
        "firm_name"
    )
)

gold_repeat_club_summary.show(5,truncate=False)

StatementMeta(, 347746e8-be41-4dab-b918-81cb31aedd95, 30, Finished, Available, Finished, False)

+--------+--------------+---------+---------------+------------+-----------+----------------+
|sap_code|firm_name     |club_name|periods_in_club|first_period|last_period|elgibilty_values|
+--------+--------------+---------+---------------+------------+-----------+----------------+
|9000001 |Demo Firm 0001|SILVER   |4              |2025-11-01  |2026-09-30 |1               |
|9000002 |Demo Firm 0002|SILVER   |4              |2025-11-01  |2026-09-30 |1               |
|9000003 |Demo Firm 0003|SILVER   |4              |2025-11-01  |2026-09-30 |2               |
|9000004 |Demo Firm 0004|SILVER   |4              |2025-11-01  |2026-09-30 |2               |
|9000005 |Demo Firm 0005|BRONZE   |4              |2025-11-01  |2026-09-30 |1               |
+--------+--------------+---------+---------------+------------+-----------+----------------+
only showing top 5 rows
